# Distances computed in 4 ways

##


In [1]:
import math
from pyproj import Geod, Transformer

# (lon, lat). I'll choose 2 POI in Almaty
a = (76.8920, 43.2018)   # Mega Alma-Ata
b = (76.9566, 43.2333)   # Dostyk Plaza

def haversine(lon1, lat1, lon2, lat2, R=6_371_008.8):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    h = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(h))

def euclid(epsg):
    tr = Transformer.from_crs("EPSG:4326", epsg, always_xy=True)
    (x1, y1), (x2, y2) = tr.transform(*a), tr.transform(*b)
    return math.hypot(x2 - x1, y2 - y1)

_, _, d_geod = Geod(ellps="WGS84").inv(*a, *b)

results = {
    "(a) haversine":           haversine(*a, *b),
    "(b) Geod: the reference": d_geod,
    "(c) Euclid EPSG:3857":    euclid("EPSG:3857"),
    "(d) Euclid UTM 43N":      euclid("EPSG:32643"),
}
for name, d in results.items():
    print(f"{name:26s} {d:9.1f} m  {100*(d/d_geod - 1):+6.2f}%")

(a) haversine                 6298.6 m   -0.16%
(b) Geod: the reference       6308.6 m   +0.00%
(c) Euclid EPSG:3857          8652.5 m  +37.16%
(d) Euclid UTM 43N            6307.9 m   -0.01%


### now I want to explain why this results appears this way and try to answer the questions